# SMS Spam Detection Using Machine Learning

This project implements an SMS spam detection system using machine learning.

### Methodology

1. **Pre-processing**
   - Cue-preserving text cleaning (ALL-CAPS, URLs, numbers by digit length, ! ? £ $ kept as marker tokens)
   - Lowercasing and punctuation removal
   - Tokenisation
   - Stopword removal (spam-indicative words kept)
   - Stemming

2. **Feature Representation**
   - Bag of Words (BoW)
   - TF-IDF
   - Unigrams and bigrams

3. **Model Training**
   - Naive Bayes
   - Support Vector Machine (SVM)
   - Logistic Regression
   - Random Forest
   - Grid Search with 10-fold cross-validation

4. **Evaluation**
   - Accuracy, precision, recall and F1-score
   - Confusion matrices
   - Important features
   - Wilcoxon signed-rank test

### Dataset

The **UCI SMS Spam Collection** dataset contains 5,574 SMS messages:

- 4,827 ham messages
- 747 spam messages

Dataset: [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection)

The dataset file is `spam.csv`, with `v1` as the label and `v2` as the message text.

In [9]:
# Libraries needed
import argparse
import json
import re
import string
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from scipy.stats import wilcoxon

try:
    import nltk
    from nltk.corpus import stopwords as nltk_stopwords
    from nltk.stem import PorterStemmer
    from nltk.tokenize import word_tokenize as nltk_word_tokenize
    NLTK_AVAILABLE = True
except ImportError:
    NLTK_AVAILABLE = False

warnings.filterwarnings("ignore")

# Minimal fallback stopword list used only if NLTK / its corpora are unavailable. If no internet access to run nltk.download. Prefer NLTK when possible.
FALLBACK_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by", "can",
    "did", "do", "does", "doing", "for", "from", "had", "has", "have",
    "having", "he", "her", "here", "hers", "herself", "him", "himself",
    "his", "how", "i", "if", "in", "into", "is", "it", "its", "itself",
    "just", "me", "more", "most", "my", "myself", "no", "nor", "not",
    "of", "on", "or", "our", "ours", "ourselves", "out", "over", "own",
    "s", "same", "she", "should", "so", "some", "such", "t", "than",
    "that", "the", "their", "theirs", "them", "themselves", "then",
    "there", "these", "they", "this", "those", "through", "to", "too",
    "under", "until", "up", "very", "was", "we", "were", "what", "when",
    "where", "which", "while", "who", "whom", "why", "will", "with",
    "you", "your", "yours", "yourself", "yourselves", "am", "or",
}

RANDOM_SEED = 42
N_FOLDS = 10
TEST_SIZE = 0.20

# Spam-indicative tokens retained despite being generic stop words (Delany et al., 2012)
KEEP_TOKENS = {"free", "win", "won", "urgent", "prize", "cash", "claim", "call", "txt"}

In [3]:
# --------------------------------------------------------------------------- #
# Stage 1: Pre-processing
# --------------------------------------------------------------------------- #
def ensure_nltk_resources():
    """Attempt to download required NLTK corpora. Safe to fail silently —
    build_preprocessor() falls back to a lightweight built-in tokenizer/
    stopword list if NLTK or its data is unavailable (e.g. no internet)."""
    if not NLTK_AVAILABLE:
        return False
    ok = True
    for pkg in ["punkt", "punkt_tab", "stopwords"]:
        try:
            nltk.data.find(f"tokenizers/{pkg}" if "punkt" in pkg else f"corpora/{pkg}")
        except LookupError:
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                ok = False
    return ok


def _simple_stem(token: str) -> str:
    """Lightweight Porter-style suffix stripping used only as a fallback
    when NLTK's PorterStemmer is unavailable."""
    for suffix in ("ing", "edly", "ed", "ies", "es", "s", "ly"):
        if token.endswith(suffix) and len(token) - len(suffix) > 2:
            return token[: -len(suffix)]
    return token


# Marker tokens that carry the stylistic / structural cues of spam. They are
# added to the text *before* punctuation, digits and URLs are removed, so the
# information survives as ordinary vocabulary items and passes through
# stop-word removal and stemming unchanged.
MARKER_TOKENS = {
    "capstok",        # an ALL-CAPS word was present (e.g. "FREE", "URGENT")
    "urltok",         # a URL was present
    "numshorttok",    # number of 1-4 digits  (e.g. prices, "150p", "2")
    "numcodetok",     # number of 5-6 digits  (e.g. SMS short codes "80062", "87066")
    "numphonetok",    # number of 7+ digits   (e.g. phone numbers, claim codes)
    "exclamtok",      # "!"
    "questiontok",    # "?"
    "currtok",        # "£" or "$"
}


def build_preprocessor():
    nltk_ready = ensure_nltk_resources()
    use_nltk = NLTK_AVAILABLE and nltk_ready

    if use_nltk:
        try:
            stop_words = set(nltk_stopwords.words("english")) - KEEP_TOKENS
            stemmer = PorterStemmer()
            tokenize = nltk_word_tokenize
            stem = stemmer.stem
        except Exception:
            use_nltk = False

    if not use_nltk:
        stop_words = FALLBACK_STOPWORDS - KEEP_TOKENS
        tokenize = str.split
        stem = _simple_stem

    url_re = re.compile(r"http\S+|www\.\S+")
    num_re = re.compile(r"\d+")
    caps_re = re.compile(r"\b[A-Z]{2,}\b")
    punct_table = str.maketrans(
        "", "", string.punctuation.replace("£", "").replace("$", "")
    )

    def _number_marker(match):
        n_digits = len(match.group(0))
        if n_digits <= 4:
            return " numshorttok "
        if n_digits <= 6:
            return " numcodetok "
        return " numphonetok "

    def clean(text: str) -> str:
        # 1. Mark cues that lowercasing / stripping would otherwise destroy
        text = caps_re.sub(lambda m: m.group(0) + " capstok ", text)   # ALL-CAPS words
        text = text.lower()
        text = url_re.sub(" urltok ", text)                            # URLs
        text = (text.replace("!", " exclamtok ")                       # ! ? £ $
                    .replace("?", " questiontok ")
                    .replace("£", " currtok ")
                    .replace("$", " currtok "))
        text = text.translate(punct_table)                             # remaining punctuation
        text = num_re.sub(_number_marker, text)                        # numbers, by digit length

        # 2. Standard normalisation
        tokens = tokenize(text)
        tokens = [t for t in tokens if t in MARKER_TOKENS or (t not in stop_words and len(t) > 1)]
        tokens = [t if t in MARKER_TOKENS else stem(t) for t in tokens]
        return " ".join(tokens)

    return clean

In [4]:
# --------------------------------------------------------------------------- #
# Stage 2: Feature representations
# --------------------------------------------------------------------------- #
def get_representations():
    return {
        "BoW_unigram":     CountVectorizer(ngram_range=(1, 1), min_df=2),
        "BoW_uni+bi":      CountVectorizer(ngram_range=(1, 2), min_df=2),
        "TFIDF_unigram":   TfidfVectorizer(ngram_range=(1, 1), min_df=2),
        "TFIDF_uni+bi":    TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    }


In [5]:
# --------------------------------------------------------------------------- #
# Stage 3: Classifiers and hyperparameter grids
# --------------------------------------------------------------------------- #
def get_classifiers():
    return {
        "NaiveBayes": (
            MultinomialNB(),
            {"alpha": [0.1, 0.5, 1.0]}
        ),
        "SVM": (
            SVC(kernel="linear", probability=True, random_state=RANDOM_SEED),
            {"C": [0.1, 1, 10]}
        ),
        "LogisticRegression": (
            LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
            {"C": [0.1, 1, 10], "penalty": ["l2"]}
        ),
        "RandomForest": (
            RandomForestClassifier(random_state=RANDOM_SEED),
            {"n_estimators": [200, 400], "max_depth": [None, 30]}
        ),
    }

In [6]:
# --------------------------------------------------------------------------- #
# Stage 4: Evaluation
# --------------------------------------------------------------------------- #
def false_positive_rate(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return fp / (fp + tn) if (fp + tn) > 0 else 0.0


def evaluate(y_true, y_pred, y_score):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score) if y_score is not None else np.nan,
        "fpr": false_positive_rate(y_true, y_pred),
    }


def top_features(vectorizer, clf, clf_name, n=20):
    """Return the n words/bigrams most associated with the spam class."""
    feature_names = np.array(vectorizer.get_feature_names_out())
    if clf_name == "NaiveBayes":
        scores = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
    elif clf_name in ("SVM", "LogisticRegression"):
        coef = clf.coef_
        scores = coef.toarray().ravel() if hasattr(coef, "toarray") else np.ravel(coef)
    elif clf_name == "RandomForest":
        scores = clf.feature_importances_
    else:
        return []
    top_idx = np.argsort(scores)[-n:][::-1]
    return list(zip(feature_names[top_idx], scores[top_idx]))

In [7]:
# --------------------------------------------------------------------------- #
# Main experiment loop
# --------------------------------------------------------------------------- #
def run_experiments(df: pd.DataFrame, outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)

    print("Stage 1: pre-processing...")
    clean = build_preprocessor()
    df["clean_text"] = df["text"].apply(clean)

    X_train_text, X_test_text, y_train, y_test = train_test_split(
        df["clean_text"], df["label"], test_size=TEST_SIZE,
        stratify=df["label"], random_state=RANDOM_SEED
    )

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    representations = get_representations()
    classifiers = get_classifiers()

    results = []
    cv_f1_scores = {}          # for Wilcoxon test: config -> list of per-fold F1
    fitted_artifacts = {}      # config -> (vectorizer, best_estimator) for explainability

    print("Stage 2-3: feature extraction + model training/tuning (16 configurations)...")
    for rep_name, vectorizer_template in representations.items():
        for clf_name, (base_clf, param_grid) in classifiers.items():
            config_name = f"{clf_name}__{rep_name}"
            print(f"  -> {config_name}")

            vectorizer = vectorizer_template.__class__(**vectorizer_template.get_params())
            X_train_vec = vectorizer.fit_transform(X_train_text)
            X_test_vec = vectorizer.transform(X_test_text)

            grid = GridSearchCV(
                base_clf, param_grid, scoring="f1", cv=skf, n_jobs=-1
            )
            grid.fit(X_train_vec, y_train)
            best_clf = grid.best_estimator_

            # Collect per-fold F1 for the best hyperparameters (for significance testing)
            fold_f1 = []
            for train_idx, val_idx in skf.split(X_train_vec, y_train):
                clf_fold = best_clf.__class__(**best_clf.get_params())
                clf_fold.fit(X_train_vec[train_idx], y_train.iloc[train_idx])
                pred_fold = clf_fold.predict(X_train_vec[val_idx])
                fold_f1.append(f1_score(y_train.iloc[val_idx], pred_fold, zero_division=0))
            cv_f1_scores[config_name] = fold_f1

            # Final held-out test evaluation
            y_pred = best_clf.predict(X_test_vec)
            if hasattr(best_clf, "predict_proba"):
                y_score = best_clf.predict_proba(X_test_vec)[:, 1]
            else:
                y_score = None
            metrics = evaluate(y_test, y_pred, y_score)
            metrics.update({
                "classifier": clf_name,
                "representation": rep_name,
                "config": config_name,
                "best_params": grid.best_params_,
                "cv_f1_mean": float(np.mean(fold_f1)),
                "cv_f1_std": float(np.std(fold_f1)),
            })
            results.append(metrics)

            fitted_artifacts[config_name] = (vectorizer, best_clf, y_test, y_pred)

    results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
    results_df.to_csv(outdir / "configuration_results.csv", index=False)
    print("\nTop 5 configurations by test F1-score:")
    print(results_df[["config", "accuracy", "precision", "recall", "f1", "roc_auc", "fpr"]].head(5).to_string(index=False))

    # ---------------- Stage 4a: statistical significance (Wilcoxon) ---------------- #
    best_config = results_df.iloc[0]["config"]
    print(f"\nStage 4: statistical comparison of top configurations against best ({best_config})...")
    sig_results = {}
    for config_name, fold_scores in cv_f1_scores.items():
        if config_name == best_config:
            continue
        try:
            stat, p_value = wilcoxon(cv_f1_scores[best_config], fold_scores)
            sig_results[config_name] = {"statistic": float(stat), "p_value": float(p_value)}
        except ValueError:
            sig_results[config_name] = {"statistic": None, "p_value": None}
    with open(outdir / "significance_tests.json", "w") as f:
        json.dump({"reference_config": best_config, "comparisons": sig_results}, f, indent=2)

    # ---------------- Stage 4b: confusion matrix + top features for best config ---------------- #
    vec_best, clf_best, y_true_best, y_pred_best = fitted_artifacts[best_config]
    cm = confusion_matrix(y_true_best, y_pred_best, labels=[0, 1])
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix – {best_config}")
    plt.tight_layout()
    plt.savefig(outdir / "confusion_matrix_best.png", dpi=200)
    plt.close()

    clf_short = best_config.split("__")[0]
    features = top_features(vec_best, clf_best, clf_short, n=20)
    pd.DataFrame(features, columns=["feature", "score"]).to_csv(
        outdir / "top_features_best_config.csv", index=False
    )

    # ---------------- Summary bar chart across all configurations ---------------- #
    plt.figure(figsize=(10, 5))
    order = results_df.sort_values("f1", ascending=False)
    ax = sns.barplot(data=order, x="config", y="f1", hue="representation", dodge=False)
    # Zoom y-axis to the data range so small F1 differences are visible
    ymin = max(0.0, order["f1"].min() - 0.015)
    ymax = min(1.0, order["f1"].max() + 0.01)
    ax.set_ylim(ymin, ymax)
    for bar in ax.patches:
        height = bar.get_height()
        if height > 0:
            ax.annotate(f"{height:.3f}",
                        (bar.get_x() + bar.get_width() / 2, height),
                        ha="center", va="bottom", fontsize=7, rotation=90,
                        xytext=(0, 3), textcoords="offset points")
    plt.xticks(rotation=75, ha="right")
    plt.ylabel("Test F1-score")
    plt.title("F1-score by Classifier–Representation Configuration")
    ax.legend(title="representation", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(outdir / "f1_comparison_all_configs.png", dpi=200, bbox_inches="tight")
    plt.close()

    print(f"\nAll outputs written to: {outdir.resolve()}")
    return results_df


def load_dataset(path: str) -> pd.DataFrame:
    """Loads the UCI SMS Spam Collection CSV (v1=label, v2=text) or a generic
    2-column {label,text} CSV, and maps labels to {ham:0, spam:1}."""
    raw = pd.read_csv(path, encoding="latin-1")
    if {"v1", "v2"}.issubset(raw.columns):
        df = raw[["v1", "v2"]].rename(columns={"v1": "label", "v2": "text"})
    elif {"label", "text"}.issubset(raw.columns):
        df = raw[["label", "text"]]
    else:
        # fall back: assume first two columns are label, text
        df = raw.iloc[:, :2]
        df.columns = ["label", "text"]
    df["label"] = df["label"].str.strip().str.lower().map({"ham": 0, "spam": 1})
    df = df.dropna(subset=["label", "text"]).reset_index(drop=True)
    df["label"] = df["label"].astype(int)
    return df

In [8]:
np.random.seed(RANDOM_SEED)

DATA_PATH = "spam.csv"
OUTPUT_DIR = "results"

dataset = load_dataset(DATA_PATH)

print(f"Loaded {len(dataset)} messages "
      f"({(dataset['label'] == 1).sum()} spam, "
      f"{(dataset['label'] == 0).sum()} ham)")

run_experiments(dataset, Path(OUTPUT_DIR))

Loaded 5572 messages (747 spam, 4825 ham)
Stage 1: pre-processing...
Stage 2-3: feature extraction + model training/tuning (16 configurations)...
  -> NaiveBayes__BoW_unigram
  -> SVM__BoW_unigram
  -> LogisticRegression__BoW_unigram
  -> RandomForest__BoW_unigram
  -> NaiveBayes__BoW_uni+bi
  -> SVM__BoW_uni+bi
  -> LogisticRegression__BoW_uni+bi
  -> RandomForest__BoW_uni+bi
  -> NaiveBayes__TFIDF_unigram
  -> SVM__TFIDF_unigram
  -> LogisticRegression__TFIDF_unigram
  -> RandomForest__TFIDF_unigram
  -> NaiveBayes__TFIDF_uni+bi
  -> SVM__TFIDF_uni+bi
  -> LogisticRegression__TFIDF_uni+bi
  -> RandomForest__TFIDF_uni+bi

Top 5 configurations by test F1-score:
                           config  accuracy  precision   recall       f1  roc_auc      fpr
  LogisticRegression__BoW_unigram  0.990135   0.992857 0.932886 0.961938 0.988113 0.001035
   LogisticRegression__BoW_uni+bi  0.989238   0.992806 0.926174 0.958333 0.989794 0.001035
                SVM__TFIDF_uni+bi  0.988341   0.959459 0.

,accuracy,precision,recall,f1,roc_auc,fpr,classifier,representation,config,best_params,cv_f1_mean,cv_f1_std
2,0.990135,0.992857,0.932886,0.961938,0.988113,0.001035,LogisticRegression,BoW_unigram,LogisticRegression__BoW_unigram,"{'C': 10, 'penalty': 'l2'}",0.954112,0.021965
6,0.989238,0.992806,0.926174,0.958333,0.989794,0.001035,LogisticRegression,BoW_uni+bi,LogisticRegression__BoW_uni+bi,"{'C': 10, 'penalty': 'l2'}",0.948656,0.020314
13,0.988341,0.959459,0.953020,0.956229,0.992879,0.006211,SVM,TFIDF_uni+bi,SVM__TFIDF_uni+bi,{'C': 1},0.951335,0.018723
5,0.988341,0.985714,0.926174,0.955017,0.992351,0.002070,SVM,BoW_uni+bi,SVM__BoW_uni+bi,{'C': 0.1},0.948418,0.019291
10,0.987444,0.972028,0.932886,0.952055,0.990023,0.004141,LogisticRegression,TFIDF_unigram,LogisticRegression__TFIDF_unigram,"{'C': 10, 'penalty': 'l2'}",0.943935,0.018297
14,0.987444,0.972028,0.932886,0.952055,0.990801,0.004141,LogisticRegression,TFIDF_uni+bi,LogisticRegression__TFIDF_uni+bi,"{'C': 10, 'penalty': 'l2'}",0.942318,0.020107
1,0.987444,0.985612,0.919463,0.951389,0.988703,0.002070,SVM,BoW_unigram,SVM__BoW_unigram,{'C': 1},0.952729,0.019301
4,0.986547,0.952703,0.946309,0.949495,0.984423,0.007246,NaiveBayes,BoW_uni+bi,NaiveBayes__BoW_uni+bi,{'alpha': 0.1},0.956405,0.013729
12,0.986547,0.965278,0.932886,0.948805,0.988571,0.005176,NaiveBayes,TFIDF_uni+bi,NaiveBayes__TFIDF_uni+bi,{'alpha': 0.1},0.955646,0.016053
8,0.986547,0.992647,0.906040,0.947368,0.986549,0.001035,NaiveBayes,TFIDF_unigram,NaiveBayes__TFIDF_unigram,{'alpha': 0.1},0.953605,0.017605


In [15]:
"""Supplementary analysis for the Results chapter. Reads configuration_results.csv, significance_tests.json and top_features_best_config.csv (set IN below) and writes enriched.csv, holm.csv and figs/*.png."""
import os; IN="/mnt/user-data/uploads"; os.makedirs("figs",exist_ok=True)
import pandas as pd, numpy as np, json
from scipy import stats
df=pd.read_csv("results/configuration_results.csv")
NS,NH=149,966
df['TP']=(df.recall*NS).round().astype(int); df['FN']=NS-df.TP
df['FP']=(df.fpr*NH).round().astype(int); df['TN']=NH-df.FP
# check
assert np.allclose((df.TP+df.TN)/(NS+NH),df.accuracy,atol=1e-6)
rng=np.random.default_rng(42)
lo=[];hi=[]
for _,r in df.iterrows():
    p=np.array([r.TP,r.FN,r.FP,r.TN])/(NS+NH)
    s=rng.multinomial(NS+NH,p,size=10000)
    f1=2*s[:,0]/(2*s[:,0]+s[:,1]+s[:,2])
    lo.append(np.percentile(f1,2.5));hi.append(np.percentile(f1,97.5))
df['f1_lo']=lo;df['f1_hi']=hi
t=stats.t.ppf(0.975,9)
# note: cv_f1_std from np.std (ddof=0) -> convert to sample sd
df['cv_sd']=df.cv_f1_std*np.sqrt(10/9)
df['cv_ci']=t*df.cv_sd/np.sqrt(10)
df.to_csv('enriched.csv',index=False)
pd.set_option('display.width',250)
print(df[['config','TP','FN','FP','TN','f1','f1_lo','f1_hi','cv_f1_mean','cv_ci']].round(4))
print('rho test vs cv F1',stats.spearmanr(df.f1,df.cv_f1_mean))
print(df.groupby('classifier')[['f1','precision','recall','fpr','roc_auc','cv_f1_mean']].mean().round(4))
print(df.groupby('representation')[['f1','precision','recall','fpr','roc_auc','cv_f1_mean']].mean().round(4))
# variance decomposition
gm=df.f1.mean(); sst=((df.f1-gm)**2).sum()
ssc=sum(4*(g.f1.mean()-gm)**2 for _,g in df.groupby('classifier'))
ssr=sum(4*(g.f1.mean()-gm)**2 for _,g in df.groupby('representation'))
print('SST',sst,'class %',ssc/sst,'rep %',ssr/sst,'resid %',1-(ssc+ssr)/sst)
sig=json.load(open('results/significance_tests.json'))['comparisons']
p=pd.Series({k:v['p_value'] for k,v in sig.items()}).sort_values()
m=len(p); holm=[];prev=0
for i,(k,v) in enumerate(p.items()):
    a=min(1,(m-i)*v); prev=max(prev,a); holm.append(prev)
res=pd.DataFrame({'p':p,'holm':holm}); print(res.round(4)); res.to_csv('holm.csv')
ref=df[df.config=='LogisticRegression__BoW_unigram'].iloc[0]
d=(ref.cv_f1_mean-df.set_index('config').cv_f1_mean)
print((d.loc[res.index]).round(4))
print('FP=1 configs',df[df.FP==1].config.tolist(), 'min FP',df.FP.min(), df.FP.max())
print('Recall range',df.recall.min(),df.recall.max(),'prec',df.precision.min(),df.precision.max(),'acc',df.accuracy.min(),df.accuracy.max(),'auc',df.roc_auc.min(),df.roc_auc.max(),df.loc[df.roc_auc.idxmax(),'config'])
print('F1 range',df.f1.min(),df.f1.max(),'cv range',df.cv_f1_mean.min(),df.cv_f1_mean.max(), df.loc[df.cv_f1_mean.idxmax(),'config'])
print('cv_sd range',df.cv_sd.min(),df.cv_sd.max())
print(df.sort_values('cv_f1_mean',ascending=False)[['config','cv_f1_mean']].head(5))
import pandas as pd, numpy as np, json
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.spines.top':False,'axes.spines.right':False,'axes.titlesize':10,'axes.labelsize':9,'savefig.dpi':300,'figure.dpi':150})
df=pd.read_csv('enriched.csv')
CC={'LogisticRegression':'#1f4e79','SVM':'#2a9d8f','NaiveBayes':'#e9a23b','RandomForest':'#b5483a'}
CN={'LogisticRegression':'Logistic Regression','SVM':'SVM (linear)','NaiveBayes':'Naïve Bayes','RandomForest':'Random Forest'}
RN={'BoW_unigram':'BoW (uni)','BoW_uni+bi':'BoW (uni+bi)','TFIDF_unigram':'TF-IDF (uni)','TFIDF_uni+bi':'TF-IDF (uni+bi)'}
MK={'BoW_unigram':'o','BoW_uni+bi':'s','TFIDF_unigram':'^','TFIDF_uni+bi':'D'}
df['label']=df.classifier.map(CN)+' | '+df.representation.map(RN)
d=df.sort_values('f1')
# Fig1
fig,ax=plt.subplots(figsize=(6.3,5.4))
for i,(_,r) in enumerate(d.iterrows()):
    ax.errorbar(r.f1,i,xerr=[[r.f1-r.f1_lo],[r.f1_hi-r.f1]],fmt=MK[r.representation],color=CC[r.classifier],capsize=2.5,ms=5.5,lw=1.2)
    ax.text(r.f1_hi+0.002,i,f'{r.f1:.3f}',va='center',fontsize=7.5)
ax.set_yticks(range(len(d)));ax.set_yticklabels(d.label,fontsize=8)
ax.set_xlabel('Test F1-score (spam class), point estimate with 95% bootstrap CI');ax.set_xlim(0.885,0.995)
ax.grid(axis='x',alpha=.3)
h=[Line2D([],[],color=c,marker='o',ls='',label=CN[k]) for k,c in CC.items()]+[Line2D([],[],color='grey',marker=m,ls='',label=RN[k]) for k,m in MK.items()]
fig.legend(handles=h,loc='lower center',frameon=False,fontsize=7.5,ncol=4)
plt.tight_layout(rect=(0,0.1,1,1));plt.savefig('figs/fig1_f1_ci.png');plt.close()
# Fig2 confusion
b=df[df.config=='LogisticRegression__BoW_unigram'].iloc[0]
cm=np.array([[b.TN,b.FP],[b.FN,b.TP]])
fig,axs=plt.subplots(1,2,figsize=(6.3,2.9))
for ax,mat,title,fmt in [(axs[0],cm,'(a) Counts',None),(axs[1],cm/cm.sum(1,keepdims=True)*100,'(b) Row-normalised (%)',None)]:
    im=ax.imshow(mat,cmap='Blues',vmin=0,vmax=mat.max()*(1.0 if title.startswith('(a)') else 1.0))
    for i in range(2):
        for j in range(2):
            v=mat[i,j];txt=f'{int(v)}' if title.startswith('(a)') else f'{v:.1f}%'
            ax.text(j,i,txt,ha='center',va='center',fontsize=12,fontweight='bold',color='white' if v>mat.max()*0.5 else '#1f2d3d')
    ax.set_xticks([0,1]);ax.set_xticklabels(['ham','spam']);ax.set_yticks([0,1]);ax.set_yticklabels(['ham','spam'])
    ax.set_xlabel('Predicted');ax.set_ylabel('Actual');ax.set_title(title)
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout();plt.savefig('figs/fig2_confusion.png');plt.close()
# Fig3 heatmaps
cl=['LogisticRegression','SVM','NaiveBayes','RandomForest'];rp=list(RN)
fig,axs=plt.subplots(1,2,figsize=(6.5,2.9))
for ax,col,title,cmap,fmt in [(axs[0],'f1','(a) Test F1-score','Blues','{:.3f}'),(axs[1],'fpr','(b) False positive rate (%)','Reds_r','{:.2f}')]:
    M=np.array([[df[(df.classifier==c)&(df.representation==r)][col].iloc[0]*(100 if col=='fpr' else 1) for r in rp] for c in cl])
    im=ax.imshow(M,cmap=cmap,aspect='auto')
    for i in range(4):
        for j in range(4):
            lum=(M[i,j]-M.min())/(M.max()-M.min()+1e-9)
            dark=(lum>0.55) if col=='f1' else (lum<0.45)
            ax.text(j,i,fmt.format(M[i,j]),ha='center',va='center',fontsize=8.5,color='white' if dark else '#222')
    ax.set_xticks(range(4));ax.set_xticklabels([RN[r].replace(' (','\n(') for r in rp],fontsize=7.5)
    ax.set_yticks(range(4));ax.set_yticklabels([CN[c] for c in cl],fontsize=8) if ax is axs[0] else ax.set_yticks([])
    ax.set_title(title)
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout();plt.savefig('figs/fig3_heatmaps.png');plt.close()
# Fig4 precision-recall
fig,ax=plt.subplots(figsize=(6.0,4.2))
for _,r in df.iterrows():
    ax.scatter(r.recall,r.precision,s=70,marker=MK[r.representation],color=CC[r.classifier],edgecolor='white',lw=.6,zorder=3)
ax.scatter(b.recall,b.precision,s=260,facecolor='none',edgecolor='black',lw=1.2,zorder=4)
ax.annotate('Best by F1:\nLR | BoW (uni)',(b.recall,b.precision),xytext=(b.recall+0.012,b.precision-0.007),fontsize=8,ha='left',arrowprops=dict(arrowstyle='-',lw=.6))
ax.set_xlabel('Recall (spam caught)');ax.set_ylabel('Precision (flagged messages that are spam)')
ax.grid(alpha=.3)
h=[Line2D([],[],color=c,marker='o',ls='',label=CN[k]) for k,c in CC.items()]+[Line2D([],[],color='grey',marker=m,ls='',label=RN[k]) for k,m in MK.items()]
ax.legend(handles=h,loc='lower left',frameon=False,fontsize=7.5,ncol=2)
plt.tight_layout();plt.savefig('figs/fig4_pr.png');plt.close()
# Fig5 dumbbell
d=df.sort_values('f1')
fig,ax=plt.subplots(figsize=(6.3,5.4))
for i,(_,r) in enumerate(d.iterrows()):
    ax.plot([r.cv_f1_mean,r.f1],[i,i],color='#bbb',lw=1,zorder=1)
    ax.errorbar(r.cv_f1_mean,i,xerr=r.cv_ci,fmt='o',color='#555',mfc='white',capsize=2.5,ms=5,lw=1,zorder=2)
    ax.scatter(r.f1,i,color=CC[r.classifier],s=38,zorder=3)
ax.set_yticks(range(16));ax.set_yticklabels(d.label,fontsize=8)
ax.set_xlabel('F1-score');ax.grid(axis='x',alpha=.3);ax.set_xlim(0.91,0.975)
h=[Line2D([],[],color='#555',marker='o',mfc='white',ls='-',label='10-fold CV mean ± 95% CI (training set)'),Line2D([],[],color='k',marker='o',ls='',label='Held-out test F1 (colour = classifier)')]
fig.legend(handles=h,loc='lower center',frameon=False,fontsize=8,ncol=1)
plt.tight_layout(rect=(0,0.08,1,1));plt.savefig('figs/fig5_cv_vs_test.png');plt.close()
# Fig6 p-values
h=pd.read_csv('holm.csv',index_col=0).sort_values('p',ascending=False)
fig,ax=plt.subplots(figsize=(6.3,4.6))
y=np.arange(len(h))
ax.barh(y-0.19,h.p,height=0.36,color=['#b5483a' if v<0.05 else '#9db4c8' for v in h.p],label='Raw p-value')
ax.barh(y+0.19,h.holm,height=0.36,color='#555',alpha=.55,label='Holm-adjusted p-value')
ax.axvline(0.05,color='k',ls='--',lw=1);ax.text(0.052,len(h)-0.4,'α = 0.05',fontsize=8)
lab=[]
for k in h.index:
    c,r=k.split('__');lab.append(CN[c]+' | '+RN[r])
ax.set_yticks(y);ax.set_yticklabels(lab,fontsize=7.5)
ax.set_xlabel('p-value (Wilcoxon signed-rank vs. LR | BoW (uni), paired 10-fold F1)')
hh=[plt.Rectangle((0,0),1,1,color='#b5483a',label='Raw p < 0.05'),plt.Rectangle((0,0),1,1,color='#9db4c8',label='Raw p ≥ 0.05'),plt.Rectangle((0,0),1,1,color='#555',alpha=.55,label='Holm-adjusted p')]
fig.legend(handles=hh,loc='lower center',frameon=False,fontsize=8,ncol=3);ax.grid(axis='x',alpha=.3)
plt.tight_layout(rect=(0,0.06,1,1));plt.savefig('figs/fig6_pvalues.png');plt.close()
# Fig7 features
f=pd.read_csv('results/top_features_best_config.csv').sort_values('score')
mk={'numphonetok','urltok','numcodetok','numshorttok','capstok','exclamtok','questiontok','currtok'}
fig,ax=plt.subplots(figsize=(6.0,4.6))
ax.barh(f.feature,f.score,color=['#b5483a' if x in mk else '#1f4e79' for x in f.feature])
for i,v in enumerate(f.score): ax.text(v+0.05,i,f'{v:.2f}',va='center',fontsize=7.5)
ax.set_xlabel('Logistic regression coefficient (log-odds contribution towards spam)')
ax.set_xlim(0,6.1)
h=[Line2D([],[],color='#b5483a',lw=6,label='Engineered marker token'),Line2D([],[],color='#1f4e79',lw=6,label='Lexical (stemmed) term')]
ax.legend(handles=h,loc='lower right',frameon=False,fontsize=8)
plt.tight_layout();plt.savefig('figs/fig7_features.png');plt.close()
print(df[['config','f1','f1_lo','f1_hi']].round(4).to_string())

                               config   TP  FN  FP   TN      f1   f1_lo   f1_hi  cv_f1_mean   cv_ci
0     LogisticRegression__BoW_unigram  139  10   1  965  0.9619  0.9375  0.9826      0.9541  0.0166
1      LogisticRegression__BoW_uni+bi  138  11   1  965  0.9583  0.9324  0.9799      0.9487  0.0153
2                   SVM__TFIDF_uni+bi  142   7   6  960  0.9562  0.9297  0.9781      0.9513  0.0141
3                     SVM__BoW_uni+bi  138  11   2  964  0.9550  0.9281  0.9771      0.9484  0.0145
4   LogisticRegression__TFIDF_unigram  139  10   4  962  0.9521  0.9249  0.9747      0.9439  0.0138
5    LogisticRegression__TFIDF_uni+bi  139  10   4  962  0.9521  0.9251  0.9752      0.9423  0.0152
6                    SVM__BoW_unigram  137  12   2  964  0.9514  0.9237  0.9744      0.9527  0.0146
7              NaiveBayes__BoW_uni+bi  141   8   7  959  0.9495  0.9225  0.9740      0.9564  0.0104
8            NaiveBayes__TFIDF_uni+bi  139  10   5  961  0.9488  0.9209  0.9724      0.9556  0.0121
